# 웹캠으로 숫자 인식하기

In [ ]:
import cv2
import numpy as np
import tensorflow as tf
from pathlib import Path


In [ ]:
# ex_11에서 학습/저장한 CNN MNIST 모델을 불러온다.
model_path = Path("CNN_MNIST_20260629-1540.keras")
if not model_path.exists():
    model_path = Path("ML/CNN_MNIST_20260629-1540.keras")

model = tf.keras.models.load_model(model_path)

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("웹캠을 열 수 없습니다.")

webcam_window = "Webcam"
process_windows = ["OTSU", "EROSION", "CROP", "REVERSE_IMAGE"]
process_windows_created = False
last_prediction = None
last_confidence = 0.0

cv2.namedWindow(webcam_window, cv2.WINDOW_NORMAL)

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            print("프레임 오류")
            break

        # 카메라 화면은 flip하지 않고 정방향 그대로 보여준다.
        display_frame = frame.copy()
        height, width = display_frame.shape[:2]
        roi_size = 300
        half = roi_size // 2
        result_area_height = 150
        center_x = width // 2
        center_y = min(height // 2, max(half, height - half - result_area_height))

        x1 = max(0, center_x - half)
        y1 = max(0, center_y - half)
        x2 = min(width, center_x + half)
        y2 = min(height, center_y + half)
        roi = display_frame[y1:y2, x1:x2]

        cv2.rectangle(display_frame, (x1, y1), (x2, y2), (0, 0, 255), 2)

        if last_prediction is not None:
            text = str(last_prediction)
            confidence_text = f"{last_confidence * 100:.1f}%"

            number_scale = 2.6
            number_thickness = 6
            number_size, number_baseline = cv2.getTextSize(
                text,
                cv2.FONT_HERSHEY_SIMPLEX,
                number_scale,
                number_thickness
            )
            number_x = (width - number_size[0]) // 2
            number_y = y2 + 14 + number_size[1]

            cv2.putText(display_frame, text, (number_x, number_y), cv2.FONT_HERSHEY_SIMPLEX, number_scale, (0, 0, 0), 12, cv2.LINE_AA)
            cv2.putText(display_frame, text, (number_x, number_y), cv2.FONT_HERSHEY_SIMPLEX, number_scale, (255, 255, 255), number_thickness, cv2.LINE_AA)

            confidence_scale = 0.9
            confidence_thickness = 2
            confidence_size, _ = cv2.getTextSize(
                confidence_text,
                cv2.FONT_HERSHEY_SIMPLEX,
                confidence_scale,
                confidence_thickness
            )
            confidence_x = (width - confidence_size[0]) // 2
            confidence_y = min(height - 10, number_y + number_baseline + confidence_size[1] + 8)

            cv2.putText(display_frame, confidence_text, (confidence_x, confidence_y), cv2.FONT_HERSHEY_SIMPLEX, confidence_scale, (0, 0, 0), 6, cv2.LINE_AA)
            cv2.putText(display_frame, confidence_text, (confidence_x, confidence_y), cv2.FONT_HERSHEY_SIMPLEX, confidence_scale, (255, 255, 255), confidence_thickness, cv2.LINE_AA)

        cv2.imshow(webcam_window, display_frame)

        key = cv2.waitKeyEx(1)
        if key == 27:  # ESC
            break

        if key in (ord('c'), ord('C'), ord('ㅊ')):
            gray_image = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
            cv2.imwrite("gray_image.png", gray_image)

            gaussian_blur = cv2.GaussianBlur(gray_image, (5, 5), 3)
            _, otsu_thresh = cv2.threshold(
                gaussian_blur,
                0,
                255,
                cv2.THRESH_BINARY + cv2.THRESH_OTSU
            )

            kernel = np.ones((5, 5), np.uint8)
            erosion = cv2.erode(otsu_thresh, kernel, iterations=5)
            cv2.imwrite("digit_binary_image.png", erosion)

            crop = erosion.copy()
            reversed_image = cv2.bitwise_not(crop)
            cv2.imwrite("IMAGE_FOR_TEST.png", reversed_image)

            model_image = cv2.resize(reversed_image, (28, 28), interpolation=cv2.INTER_AREA)
            model_image = model_image.astype("float32") / 255.0
            model_image = model_image.reshape(1, 28, 28, 1)

            probabilities = model.predict(model_image, verbose=0)[0]
            last_prediction = int(np.argmax(probabilities))
            last_confidence = float(np.max(probabilities))
            print(f"인식 결과: {last_prediction} ({last_confidence * 100:.1f}%)")

            if not process_windows_created:
                for window_name in process_windows:
                    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
                process_windows_created = True

            # 같은 imshow 윈도우 이름을 계속 사용해서 캡처할 때마다 창을 재사용한다.
            cv2.imshow("OTSU", otsu_thresh)
            cv2.imshow("EROSION", erosion)
            cv2.imshow("CROP", crop)
            cv2.imshow("REVERSE_IMAGE", reversed_image)
finally:
    cap.release()
    cv2.destroyAllWindows()
    for _ in range(5):
        cv2.waitKey(1)
